# 종합 실습 - 정답

00~04에서 배운 내용의 정답 코드입니다.

## 0. 환경 설정

In [5]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-5.4-mini")
print("\u2713 모델 준비 완료")

✓ 모델 준비 완료


---
## 1단계: 메시지와 invoke (01 복습)

- Q1: `SystemMessage`, `HumanMessage`
- Q2: `invoke`

In [6]:
from langchain.messages import SystemMessage, HumanMessage

# Q1. SystemMessage, HumanMessage
messages = [
    SystemMessage(content="당신은 한국 역사 전문가입니다."),
    HumanMessage(content="세종대왕의 업적을 3가지 알려주세요."),
]

# Q2. invoke
response = model.invoke(messages)
print(response.content[:200])

세종대왕의 대표적인 업적 3가지는 다음과 같습니다.

1. **훈민정음 창제**  
   백성이 쉽게 글을 읽고 쓸 수 있도록 한글, 즉 훈민정음을 만들었습니다.

2. **과학 기술 발전**  
   혼천의, 앙부일구, 자격루 같은 과학 기구를 제작하게 하여 천문·시간 측정 기술을 발전시켰습니다.

3. **문화와 학문 진흥**  
   집현전을 중심으로


---
## 2단계: 스트리밍 (01 복습)

- Q3: `stream`

In [7]:
# Q3. stream
print("스트리밍: ", end="")
for chunk in model.stream("대한민국의 수도는 어디인가요?"):
    print(chunk.content, end="", flush=True)
print()

스트리밍: 대한민국의 수도는 **서울특별시**입니다.


---
## 3단계: 도구 만들기 (02 복습)

- Q4: `@tool`

In [8]:
from langchain.tools import tool

# Q4. @tool
@tool
def subtract(a: int, b: int) -> int:
    """두 수를 뺍니다."""
    return a - b

@tool
def divide(a: int, b: int) -> float:
    """두 수를 나눕니다."""
    return a / b

print(f"도구 이름: {subtract.name}, {divide.name}")

도구 이름: subtract, divide


---
## 4단계: 에이전트 생성 (02 복습)

- Q5: `create_agent`
- Q6: `invoke`

In [9]:
from langchain.agents import create_agent

# Q5. create_agent
agent = create_agent(
    model=model,
    tools=[subtract, divide],
    system_prompt="당신은 계산 도우미입니다.",
)

# Q6. invoke
result = agent.invoke({
    "messages": [{"role": "user", "content": "100에서 37을 빼주세요."}]
})
print("결과:", result["messages"][-1].content)

결과: 100에서 37을 빼면 **63**입니다.


---
## 5단계: 메모리 에이전트 (03 복습)

- Q7: `InMemorySaver()`
- Q8: `thread_id`

In [10]:
from langgraph.checkpoint.memory import InMemorySaver

# Q7. InMemorySaver()
memory_agent = create_agent(
    model=model,
    tools=[subtract, divide],
    checkpointer=InMemorySaver(),
)

# Q8. thread_id
config = {"configurable": {"thread_id": "practice-1"}}

# 첫 번째 질문
r1 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "200 나누기 8은?"}]},
    config=config,
)
print("1차:", r1["messages"][-1].content)

# 두 번째 질문 - 이전 결과를 기억해야 함
r2 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "그 결과에서 10을 빼주세요."}]},
    config=config,
)
print("2차:", r2["messages"][-1].content)

1차: 200 ÷ 8 = 25
2차: 15


---
## 6단계: LangGraph 워크플로 (04 복습)

- Q9: `TypedDict`
- Q10: `add_node`, `add_edge`, `compile`

In [11]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Q9. TypedDict
class State(TypedDict):
    text: str
    char_count: int

def reverse_text(state: State) -> dict:
    return {"text": state["text"][::-1]}

def count_chars(state: State) -> dict:
    return {"char_count": len(state["text"])}

# Q10. add_node, add_edge, compile
builder = StateGraph(State)
builder.add_node("reverse", reverse_text)
builder.add_node("count", count_chars)
builder.add_edge(START, "reverse")
builder.add_edge("reverse", "count")
builder.add_edge("count", END)

graph = builder.compile()

result = graph.invoke({"text": "Hello LangGraph"})
print(f"원본 → 뒤집기: {result['text']}")
print(f"글자 수: {result['char_count']}")

원본 → 뒤집기: hparGgnaL olleH
글자 수: 15


---
## 정답 요약

| 문제 | 정답 | 출처 |
|---|---|---|
| Q1 | `SystemMessage`, `HumanMessage` | 01 - 메시지 역할 |
| Q2 | `invoke` | 01 - 동기 호출 |
| Q3 | `stream` | 01 - 스트리밍 |
| Q4 | `@tool` | 02 - 도구 정의 |
| Q5 | `create_agent` | 02 - 에이전트 생성 |
| Q6 | `invoke` | 02 - 에이전트 실행 |
| Q7 | `InMemorySaver()` | 03 - 체크포인터 |
| Q8 | `thread_id` | 03 - 세션 구분 |
| Q9 | `TypedDict` | 04 - 상태 정의 |
| Q10 | `add_node`, `add_edge`, `compile` | 04 - 그래프 빌드 |